# SportsMOT tuning for all trackers with both state estimators

In [1]:
import os
import itertools
import inspect
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime

import numpy as np
import pandas as pd
import supervision as sv

from trackers import SORTTracker, OCSORTTracker, ByteTrackTracker
from trackers.eval import evaluate_mot_sequences
from trackers.utils.state_representations import XYXYStateEstimator, XCYCSRStateEstimator

SPORTSMOT_DET_ROOT = "sportsmot_yolox_dets"
SPORTSMOT_GT_ROOT = os.path.join("TrackEval", "data", "gt", "sportsmot")

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6

STATE_ESTIMATORS = {
    "XYXY": XYXYStateEstimator,
    "XCYCSR": XCYCSRStateEstimator,
}

# Skip already-run configurations:
# - SORT + XYXY
# - ByteTrack + XYXY
# - OCSORT + XCYCSR
TRACKER_ESTIMATORS_TO_RUN = {
    "SORT": ["XCYCSR"],
    "OCSORT": ["XYXY"],
    "ByteTrack": ["XCYCSR"],
}

TRACKER_PARAM_SPACES = {
    "SORT": {
        "lost_track_buffer": [10, 30, 60],
        "track_activation_threshold": [0.25, 0.5, 0.75, 0.9],
        "minimum_consecutive_frames": [2, 3, 5],
        "minimum_iou_threshold": [0.05, 0.1, 0.3, 0.5, 0.7],
    },
    "OCSORT": {
        "lost_track_buffer": [10, 30, 60],
        "minimum_iou_threshold": [0.1, 0.3, 0.5],
        "minimum_consecutive_frames": [3, 5],
        "direction_consistency_weight": [0.0, 0.2, 0.5],
        "high_conf_det_threshold": [0.4, 0.6, 0.8],
        "delta_t": [1, 3],
    },
    "ByteTrack": {
        "lost_track_buffer": [10, 30, 60, 90],
        "track_activation_threshold": [0.2, 0.5, 0.7, 0.9],
        "minimum_consecutive_frames": [1, 3],
        "minimum_iou_threshold": [0.05, 0.1, 0.3],
        "high_conf_det_threshold": [0.5, 0.6, 0.7, 0.9],
    },
}


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def _inject_state_estimator(tracker_cls, kwargs, state_estimator_class):
    try:
        params = inspect.signature(tracker_cls.__init__).parameters
        if "state_estimator_class" in params:
            kwargs["state_estimator_class"] = state_estimator_class
    except Exception:
        pass
    return kwargs


def make_tracker(tracker_name, params_dict, state_estimator_class):
    if tracker_name == "SORT":
        kwargs = dict(
            lost_track_buffer=params_dict["lost_track_buffer"],
            track_activation_threshold=params_dict["track_activation_threshold"],
            minimum_consecutive_frames=params_dict["minimum_consecutive_frames"],
            minimum_iou_threshold=params_dict["minimum_iou_threshold"],
        )
        kwargs = _inject_state_estimator(SORTTracker, kwargs, state_estimator_class)
        return SORTTracker(**kwargs)

    if tracker_name == "OCSORT":
        kwargs = dict(
            lost_track_buffer=params_dict["lost_track_buffer"],
            minimum_iou_threshold=params_dict["minimum_iou_threshold"],
            minimum_consecutive_frames=params_dict["minimum_consecutive_frames"],
            direction_consistency_weight=params_dict["direction_consistency_weight"],
            high_conf_det_threshold=params_dict["high_conf_det_threshold"],
            delta_t=params_dict["delta_t"],
        )
        kwargs = _inject_state_estimator(OCSORTTracker, kwargs, state_estimator_class)
        return OCSORTTracker(**kwargs)

    if tracker_name == "ByteTrack":
        kwargs = dict(
            lost_track_buffer=params_dict["lost_track_buffer"],
            track_activation_threshold=params_dict["track_activation_threshold"],
            minimum_consecutive_frames=params_dict["minimum_consecutive_frames"],
            minimum_iou_threshold=params_dict["minimum_iou_threshold"],
            high_conf_det_threshold=params_dict["high_conf_det_threshold"],
        )
        kwargs = _inject_state_estimator(ByteTrackTracker, kwargs, state_estimator_class)
        return ByteTrackTracker(**kwargs)

    raise ValueError(f"Unknown tracker: {tracker_name}")


def run_one_combination(tracker_name, estimator_name, estimator_class, split, params_dict):
    tracker = make_tracker(tracker_name, params_dict, estimator_class)

    det_root = os.path.join(SPORTSMOT_DET_ROOT, split)
    gt_dir = os.path.join(SPORTSMOT_GT_ROOT, split)

    param_tag = "_".join([f"{k}-{params_dict[k]}" for k in params_dict.keys()])
    save_dir = os.path.join(
        f"{tracker_name}_outputs_sportsmot_tuning_state_estimators",
        f"{split}_{estimator_name}_{param_tag}",
    )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                left = float(left)
                top = float(top)
                right = float(right)
                bottom = float(bottom)

                # Guard against invalid numeric outputs (e.g., NaN/inf) from tracker states.
                if not np.isfinite([left, top, right, bottom]).all():
                    continue

                width = right - left
                height = bottom - top
                if width <= 0 or height <= 0:
                    continue

                tid_val = float(tid)
                if not np.isfinite(tid_val):
                    continue
                tid_int = int(tid_val)

                line = (
                    f"{int(frame_id)},{tid_int},"
                    f"{left:.1f},{top:.1f},{width:.1f},{height:.1f},"
                    "-1,-1,-1,-1\n"
                )
                output_lines.append(line)

        with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(gt_dir=gt_dir, tracker_dir=save_dir, metrics=["CLEAR", "HOTA", "Identity"])
    agg = result.to_dict()["aggregate"]
    out = {
        "tracker": tracker_name,
        "state_estimator": estimator_name,
        **params_dict,
        "HOTA": agg["HOTA"]["HOTA"],
        "IDF1": agg["Identity"]["IDF1"],
        "MOTA": agg["CLEAR"]["MOTA"],
        "output_dir": save_dir,
    }
    print(f"{tracker_name} | {estimator_name} | {params_dict} -> HOTA={out['HOTA']:.3f}")
    return out


In [2]:
def tune_tracker_for_estimators(tracker_name, split="val", max_workers=None):
    if max_workers is None:
        max_workers = os.cpu_count()

    param_space = TRACKER_PARAM_SPACES[tracker_name]
    param_keys = list(param_space.keys())
    combinations = list(itertools.product(*[param_space[k] for k in param_keys]))

    all_rows = []
    estimators_to_run = TRACKER_ESTIMATORS_TO_RUN.get(tracker_name, list(STATE_ESTIMATORS.keys()))
    for estimator_name in estimators_to_run:
        estimator_class = STATE_ESTIMATORS[estimator_name]
        print(f"\n=== {tracker_name} | {estimator_name} | {len(combinations)} combinations ===")
        ctx = mp.get_context("fork")
        with ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
            futures = []
            for comb in combinations:
                params_dict = {k: v for k, v in zip(param_keys, comb)}
                futures.append(ex.submit(run_one_combination, tracker_name, estimator_name, estimator_class, split, params_dict))

            for i, fut in enumerate(futures, start=1):
                try:
                    all_rows.append(fut.result())
                    if i % 20 == 0 or i == len(futures):
                        print(f"Completed {i}/{len(futures)}")
                except Exception as e:
                    print(f"FAILED {i}/{len(futures)}: {repr(e)}")

    df = pd.DataFrame(all_rows)
    date_str = datetime.now().strftime("%Y%m%d")
    out_csv = f"{tracker_name.lower()}_sportsmot_val_tuning_both_state_estimators_{date_str}.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
    return df


sort_sportsmot_both_df = tune_tracker_for_estimators("SORT", split="val")
ocsort_sportsmot_both_df = tune_tracker_for_estimators("OCSORT", split="val")
bytetrack_sportsmot_both_df = tune_tracker_for_estimators("ByteTrack", split="val")

summary_df = pd.concat([
    sort_sportsmot_both_df,
    ocsort_sportsmot_both_df,
    bytetrack_sportsmot_both_df,
], ignore_index=True)

best_rows = summary_df.sort_values("HOTA", ascending=False).groupby(["tracker", "state_estimator"], as_index=False).first()
best_rows.sort_values(["tracker", "state_estimator"])


=== SORT | XCYCSR | 180 combinations ===
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.1} -> HOTA=0.785
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.3} -> HOTA=0.760
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 3, 'minimum_iou_threshold': 0.5} -> HOTA=0.658
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.05} -> HOTA=0.787
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 3, 'minimum_iou_threshold': 0.1} -> HOTA=0.784
SORT | XCYCSR | {'lost_track_buffer': 10, 'track_activation_threshold': 0.25, 'minimum_consecutive_frames': 3, 'minimum_iou_threshold': 0.3} -> HOTA=0.758
SORT | XCYCSR | {'lost_trac

,tracker,state_estimator,lost_track_buffer,track_activation_threshold,minimum_consecutive_frames,minimum_iou_threshold,HOTA,IDF1,MOTA,output_dir,direction_consistency_weight,high_conf_det_threshold,delta_t
0,ByteTrack,XCYCSR,60,0.9,1,0.05,0.803475,0.798413,0.981077,ByteTrack_outputs_sportsmot_tuning_state_estim...,NaN,0.7,NaN
1,OCSORT,XYXY,60,NaN,3,0.10,0.822422,0.827481,0.977789,OCSORT_outputs_sportsmot_tuning_state_estimato...,0.2,0.6,3.0
2,SORT,XCYCSR,10,0.9,2,0.05,0.790208,0.780100,0.979382,SORT_outputs_sportsmot_tuning_state_estimators...,NaN,NaN,NaN


In [3]:
import shutil

PARAM_KEYS = {
    "SORT": ["lost_track_buffer", "track_activation_threshold", "minimum_consecutive_frames", "minimum_iou_threshold"],
    "OCSORT": ["lost_track_buffer", "minimum_iou_threshold", "minimum_consecutive_frames", "direction_consistency_weight", "high_conf_det_threshold", "delta_t"],
    "ByteTrack": ["lost_track_buffer", "track_activation_threshold", "minimum_consecutive_frames", "minimum_iou_threshold", "high_conf_det_threshold"],
}

INT_PARAMS = {"lost_track_buffer", "minimum_consecutive_frames", "delta_t"}


def generate_test_submission(tracker_name, estimator_name, estimator_class, params_dict):
    """Run tracker on SportsMOT test split and write MOT-format output files."""
    tracker = make_tracker(tracker_name, params_dict, estimator_class)
    det_root = os.path.join(SPORTSMOT_DET_ROOT, "test")

    param_tag = "_".join(f"{k}-{params_dict[k]}" for k in params_dict)
    save_dir = os.path.join(
        f"{tracker_name}_outputs_sportsmot_test_submissions",
        f"test_{estimator_name}_{param_tag}",
    )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                left, top, right, bottom = float(left), float(top), float(right), float(bottom)
                if not np.isfinite([left, top, right, bottom]).all():
                    continue
                width = right - left
                height = bottom - top
                if width <= 0 or height <= 0:
                    continue
                tid_int = int(float(tid))
                output_lines.append(
                    f"{int(frame_id)},{tid_int},{left:.1f},{top:.1f},{width:.1f},{height:.1f},-1,-1,-1,-1\n"
                )

        with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
            f.writelines(output_lines)

    print(f"  Wrote {len(os.listdir(save_dir))} sequence files to {save_dir}")
    return save_dir


print("=" * 60)
print("Generating test-set submissions from best validation params")
print("=" * 60)

submission_records = []
for _, row in best_rows.iterrows():
    tracker_name = row["tracker"]
    estimator_name = row["state_estimator"]
    estimator_class = STATE_ESTIMATORS[estimator_name]

    params_dict = {
        k: int(row[k]) if k in INT_PARAMS else float(row[k])
        for k in PARAM_KEYS[tracker_name]
    }

    print(f"\n--- {tracker_name} | {estimator_name} ---")
    print(f"  Val HOTA: {row['HOTA']:.4f} | Params: {params_dict}")

    save_dir = generate_test_submission(tracker_name, estimator_name, estimator_class, params_dict)

    zip_name = f"{tracker_name}_{estimator_name}_sportsmot_test_best"
    shutil.make_archive(zip_name, "zip", save_dir)
    print(f"  Created submission: {zip_name}.zip")

    submission_records.append({
        "tracker": tracker_name,
        "state_estimator": estimator_name,
        "val_HOTA": row["HOTA"],
        "val_IDF1": row["IDF1"],
        "val_MOTA": row["MOTA"],
        "params": str(params_dict),
        "submission_zip": zip_name + ".zip",
    })

submission_df = pd.DataFrame(submission_records)
print("\n" + "=" * 60)
print("All submissions:")
print(submission_df[["tracker", "state_estimator", "val_HOTA", "submission_zip"]].to_string(index=False))
submission_df

Generating test-set submissions from best validation params

--- ByteTrack | XCYCSR ---
  Val HOTA: 0.8035 | Params: {'lost_track_buffer': 60, 'track_activation_threshold': 0.9, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.05, 'high_conf_det_threshold': 0.7}
  Wrote 150 sequence files to ByteTrack_outputs_sportsmot_test_submissions/test_XCYCSR_lost_track_buffer-60_track_activation_threshold-0.9_minimum_consecutive_frames-1_minimum_iou_threshold-0.05_high_conf_det_threshold-0.7
  Created submission: ByteTrack_XCYCSR_sportsmot_test_best.zip

--- OCSORT | XYXY ---
  Val HOTA: 0.8224 | Params: {'lost_track_buffer': 60, 'minimum_iou_threshold': 0.1, 'minimum_consecutive_frames': 3, 'direction_consistency_weight': 0.2, 'high_conf_det_threshold': 0.6, 'delta_t': 3}
  Wrote 150 sequence files to OCSORT_outputs_sportsmot_test_submissions/test_XYXY_lost_track_buffer-60_minimum_iou_threshold-0.1_minimum_consecutive_frames-3_direction_consistency_weight-0.2_high_conf_det_threshold-0

,tracker,state_estimator,val_HOTA,val_IDF1,val_MOTA,params,submission_zip
0,ByteTrack,XCYCSR,0.803475,0.798413,0.981077,"{'lost_track_buffer': 60, 'track_activation_th...",ByteTrack_XCYCSR_sportsmot_test_best.zip
1,OCSORT,XYXY,0.822422,0.827481,0.977789,"{'lost_track_buffer': 60, 'minimum_iou_thresho...",OCSORT_XYXY_sportsmot_test_best.zip
2,SORT,XCYCSR,0.790208,0.780100,0.979382,"{'lost_track_buffer': 10, 'track_activation_th...",SORT_XCYCSR_sportsmot_test_best.zip
